In [2]:
DIGITS = ["zero","one","two","three","four","five","six","seven","eight","nine"]
LETTER_WORDS = ["go","stop","left","right","up","down","yes","no","on","off"]
CLASSES = DIGITS + LETTER_WORDS
print(len(CLASSES), CLASSES)

20 ['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'go', 'stop', 'left', 'right', 'up', 'down', 'yes', 'no', 'on', 'off']


In [3]:
WORD2CHAR = {
    "zero":"0","one":"1","two":"2","three":"3","four":"4",
    "five":"5","six":"6","seven":"7","eight":"8","nine":"9",

    "go":"A",
    "stop":"B",
    "left":"C",
    "right":"D",
    "up":"E",
    "down":"F",
    "yes":"G",
    "no":"H",
    "on":"J",
    "off":"K",
}

In [4]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

2026-04-09 11:43:32.752211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775735012.994012      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775735013.061231      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775735013.630374      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775735013.630423      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775735013.630426      55 computation_placer.cc:177] computation placer alr

In [5]:
GSC_DIR = "/kaggle/input/datasets/neehakurelli/google-speech-commands"
MAX_PER_CLASS = 800 

def list_wavs(word):
    folder = os.path.join(GSC_DIR, word)
    if not os.path.isdir(folder):
        return []
    files = [os.path.join(folder,f) for f in os.listdir(folder) if f.endswith(".wav")]
    random.shuffle(files)
    return files[:MAX_PER_CLASS]

items = []
missing = []
for w in CLASSES:
    paths = list_wavs(w)
    if len(paths) == 0:
        missing.append(w)
    print(f"{w:>5}  {len(paths)}")
    for p in paths:
        items.append((p, w))

print("TOTAL:", len(items))
print("MISSING:", missing)

 zero  800
  one  800
  two  800
three  800
 four  800
 five  800
  six  800
seven  800
eight  800
 nine  800
   go  800
 stop  800
 left  800
right  800
   up  800
 down  800
  yes  800
   no  800
   on  800
  off  800
TOTAL: 16000
MISSING: []


In [6]:

random.shuffle(items)
# Train / Val / Test split
n = len(items)

train_items = items[:int(0.8*n)]
val_items   = items[int(0.8*n):int(0.9*n)]
test_items  = items[int(0.9*n):]

print(len(train_items), len(val_items), len(test_items))

12800 1600 1600


In [7]:
SR = 16000
N_SAMPLES = SR  # 1 

N_MELS = 40
FRAME_LENGTH = 640  # 40
FRAME_STEP   = 320  # 20 ms

def decode_wav(path): #однаковий розмір входу
    audio = tf.io.read_file(path)
    wav, sr = tf.audio.decode_wav(audio, desired_channels=1)
    wav = tf.squeeze(wav, axis=-1)

    # pad/trim to 1 sec
    wav = wav[:N_SAMPLES]
    pad = N_SAMPLES - tf.shape(wav)[0]
    wav = tf.cond(pad > 0, lambda: tf.pad(wav, [[0, pad]]), lambda: wav)
    return wav

def wav_to_logmel(wav): # спектрограмa
    stft = tf.signal.stft(wav, frame_length=FRAME_LENGTH, frame_step=FRAME_STEP)
    spec = tf.abs(stft)

    mel_w = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS,
        num_spectrogram_bins=spec.shape[-1],
        sample_rate=SR,
        lower_edge_hertz=80.0,
        upper_edge_hertz=7600.0,
    )
    mel = tf.matmul(tf.square(spec), mel_w)
    logmel = tf.math.log(mel + 1e-6)
    logmel = logmel[..., tf.newaxis]  # channel for CNN
    return logmel

In [8]:
label_to_id = {w:i for i,w in enumerate(CLASSES)}
id_to_label = {i:w for w,i in label_to_id.items()}

def make_ds(items_list, batch=64, shuffle=False):
    paths = [p for p,_ in items_list]
    labels = [label_to_id[w] for _,w in items_list]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(4000, seed=SEED, reshuffle_each_iteration=True)

    def _map(path, y):
        wav = decode_wav(path)
        x = wav_to_logmel(wav)
        return x, y

    ds = ds.map(_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(train_items, batch=64, shuffle=True)
val_ds   = make_ds(val_items, batch=64, shuffle=False)
test_ds  = make_ds(test_items, batch=64, shuffle=False)

x0, y0 = next(iter(train_ds))
print("Input shape:", x0.shape, "Labels shape:", y0.shape)

2026-04-09 11:44:02.811742: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Input shape: (64, 49, 40, 1) Labels shape: (64,)


In [9]:
input_shape = x0.shape[1:]

model = keras.Sequential([
    layers.Input(shape=input_shape),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPool2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 49, 40, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 20, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 20, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 10, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         2,580 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 111,764 (436.58 KB)

 Trainable params: 111,764 (436.58 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 36s 172ms/step - accuracy: 0.0596 - loss: 2.9729 - val_accuracy: 0.1981 - val_loss: 2.5623 - learning_rate: 0.0010
Epoch 2/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 150ms/step - accuracy: 0.2324 - loss: 2.4255 - val_accuracy: 0.5487 - val_loss: 1.6086 - learning_rate: 0.0010
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 151ms/step - accuracy: 0.4872 - loss: 1.6236 - val_accuracy: 0.6681 - val_loss: 1.1291 - learning_rate: 0.0010
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.6270 - loss: 1.2042 - val_accuracy: 0.7487 - val_loss: 0.8821 - learning_rate: 0.0010
Epoch 5/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.7238 - loss: 0.9053 - val_accuracy: 0.8163 - val_loss: 0.6636 - learning_rate: 0.0010
Epoch 6/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.7683 - loss: 0.7414 - val_accuracy: 0.8138 - val_loss: 0.6143 - learning_rate: 0.0010
Epoch 7/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.8

In [10]:
test_loss, test_acc = model.evaluate(test_ds)
print("TEST accuracy:", test_acc)

25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 112ms/step - accuracy: 0.8818 - loss: 0.3813
TEST accuracy: 0.8981249928474426


In [11]:
def predict_word(path):
    wav = decode_wav(path)
    x = wav_to_logmel(wav)[tf.newaxis, ...]
    probs = model.predict(x, verbose=0)[0]
    idx = int(np.argmax(probs))
    return id_to_label[idx], float(probs[idx])

def decode_sequence(paths, conf_threshold=0.50, unknown_char="?"):
    out = []
    details = []
    for p in paths:
        w, conf = predict_word(p)
        if conf < conf_threshold or w not in WORD2CHAR:
            out.append(unknown_char)
        else:
            out.append(WORD2CHAR[w])
        details.append((os.path.basename(p), w, conf))
    return "".join(out), details

In [12]:
def random_wav_from(word):
    folder = os.path.join(GSC_DIR, word)
    f = random.choice([x for x in os.listdir(folder) if x.lower().endswith(".wav")])
    return os.path.join(folder, f)

seq_words = ["go","stop","one","two","three"] 
seq_paths = [random_wav_from(w) for w in seq_words]

text, dbg = decode_sequence(seq_paths, conf_threshold=0.50)
print("Words:", seq_words)
print("Decoded text:", text)
print("Details (file, predicted_word, conf):")
for row in dbg:
    print(row)

Words: ['go', 'stop', 'one', 'two', 'three']
Decoded text: AB123
Details (file, predicted_word, conf):
('e3b64217_nohash_0.wav', 'go', 0.9524440169334412)
('c7124b73_nohash_0.wav', 'stop', 0.995913565158844)
('3fdafe25_nohash_0.wav', 'one', 0.9995885491371155)
('918a2473_nohash_3.wav', 'two', 0.9182550311088562)
('a9abc695_nohash_0.wav', 'three', 0.9982463121414185)


In [22]:
seq_words2 = ["off","on","nine","zero","eight"]
seq_paths2 = [random_wav_from(w) for w in seq_words2]

text2, dbg2 = decode_sequence(seq_paths2, conf_threshold=0.50)
print("Words:", seq_words2)
print("Decoded text:", text2)
for row in dbg2:
    print(row)

Words: ['off', 'on', 'nine', 'zero', 'eight']
Decoded text: KJ90?
('b8c48ffb_nohash_0.wav', 'off', 0.8881592154502869)
('857366dd_nohash_0.wav', 'on', 0.8348679542541504)
('3e31dffe_nohash_4.wav', 'nine', 0.9411109089851379)
('d8c314c0_nohash_0.wav', 'zero', 0.9951381087303162)
('dbb40d24_nohash_1.wav', 'nine', 0.1685742884874344)


In [25]:
seq_words2 = ["go","stop","down","up"]
seq_paths2 = [random_wav_from(w) for w in seq_words2]

text2, dbg2 = decode_sequence(seq_paths2, conf_threshold=0.50)
print("Words:", seq_words2)
print("Decoded text:", text2)
for row in dbg2:
    print(row)

Words: ['go', 'stop', 'down', 'up']
Decoded text: A?FE
('8f0d3c27_nohash_1.wav', 'go', 0.9915544390678406)
('96d5276f_nohash_0.wav', 'stop', 0.33401429653167725)
('44260689_nohash_0.wav', 'down', 0.9702563285827637)
('2e30f9a5_nohash_0.wav', 'up', 0.6248583197593689)
